In [1]:
import boto3
import pandas as pd

BUCKET_NAME = "novaflow-saas-analytics-raw-2026-4827"
PREFIX = "raw/"
AWS_PROFILE = "novaflow"
AWS_REGION = "us-east-1"

session = boto3.Session(
    profile_name=AWS_PROFILE,
    region_name=AWS_REGION
)

s3 = session.client("s3")

print("AWS session created successfully.")

AWS session created successfully.


### List the files

In [2]:
response = s3.list_objects_v2(
    Bucket=BUCKET_NAME,
    Prefix=PREFIX
)

objects = response.get("Contents", [])

s3_inventory = pd.DataFrame(
    [
        {
            "object_key": obj["Key"],
            "size_bytes": obj["Size"],
            "last_modified": obj["LastModified"]
        }
        for obj in objects
        if obj["Key"] != PREFIX
    ]
)

s3_inventory

,object_key,size_bytes,last_modified
0,raw/experiment_assignments.csv,467623,2026-09-02 16:07:12+00:00
1,raw/experiments.csv,576,2026-09-02 16:07:12+00:00
2,raw/features.csv,756,2026-09-02 16:07:13+00:00
3,raw/marketing_attribution.csv,9395234,2026-09-02 16:07:16+00:00
4,raw/onboarding_events.csv,36092705,2026-09-02 16:07:12+00:00
5,raw/organizations.csv,596884,2026-09-02 16:07:17+00:00
6,raw/payments.csv,16496064,2026-09-02 16:07:19+00:00
7,raw/plans.csv,192,2026-09-02 16:07:21+00:00
8,raw/product_events.csv,131479843,2026-09-02 16:07:12+00:00
9,raw/sessions.csv,48402654,2026-09-02 16:07:12+00:00


### Validate the count

In [3]:
print(
    "Objects found:",
    len(s3_inventory)
)

print()

print(
    s3_inventory[
        [
            "object_key",
            "size_bytes"
        ]
    ]
)

Objects found: 13

                        object_key  size_bytes
0   raw/experiment_assignments.csv      467623
1              raw/experiments.csv         576
2                 raw/features.csv         756
3    raw/marketing_attribution.csv     9395234
4        raw/onboarding_events.csv    36092705
5            raw/organizations.csv      596884
6                 raw/payments.csv    16496064
7                    raw/plans.csv         192
8           raw/product_events.csv   131479843
9                 raw/sessions.csv    48402654
10           raw/subscriptions.csv     4586760
11         raw/support_tickets.csv     2768174
12                   raw/users.csv    10171911


### Check total cloud-storage size

In [4]:
total_size_mb = (
    s3_inventory["size_bytes"].sum()
    / (1024 ** 2)
)

print(
    f"Total raw S3 storage: {total_size_mb:,.2f} MB"
)

Total raw S3 storage: 248.39 MB


### Read one S3 file directly into Python

In [5]:
import io

response = s3.get_object(
    Bucket=BUCKET_NAME,
    Key="raw/plans.csv"
)

plans_from_s3 = pd.read_csv(
    io.BytesIO(
        response["Body"].read()
    )
)

plans_from_s3

,plan_id,plan_name,monthly_price,annual_price,max_users,storage_limit_gb,plan_status
0,P001,Free,0.0,0.0,1,5,Active
1,P002,Pro,15.0,144.0,10,100,Active
2,P003,Business,30.0,288.0,100,1000,Active


### Confirm all expected datasets

In [6]:
expected_s3_files = {
    "raw/users.csv",
    "raw/organizations.csv",
    "raw/plans.csv",
    "raw/subscriptions.csv",
    "raw/payments.csv",
    "raw/sessions.csv",
    "raw/features.csv",
    "raw/product_events.csv",
    "raw/onboarding_events.csv",
    "raw/marketing_attribution.csv",
    "raw/support_tickets.csv",
    "raw/experiments.csv",
    "raw/experiment_assignments.csv"
}

actual_s3_files = set(
    s3_inventory["object_key"]
)

missing_files = (
    expected_s3_files
    - actual_s3_files
)

print(
    "Expected files:",
    len(expected_s3_files)
)

print(
    "Files found:",
    len(
        expected_s3_files
        & actual_s3_files
    )
)

print(
    "Missing files:",
    missing_files
)

Expected files: 13
Files found: 13
Missing files: set()


## Snowflake